<a href="https://colab.research.google.com/github/chetools/CHE4061_Spring2026/blob/main/dynamic_distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!wget -N -q https://raw.githubusercontent.com/chetools/chetools/main/tools/che5.ipynb -O che5.ipynb
%run che5.ipynb

In [55]:
alpha = 2.
F = 1.
zF = 0.45
q=1.  #all liquid feed
Dsp = 0.55*F

N= 7
NFeed = 4
M0stage = 10.
M0cond = 20.
M0boil = 30.
c = 0.1  #weir constant
M0 = np.r_[M0cond, N*[M0stage], M0boil]
Minitial = np.r_[1.2*M0cond, N*[15], 1.1*M0boil]
Minitial

array([24., 15., 15., 15., 15., 15., 15., 15., 33.])

In [56]:
# 0    1     2    3       F
# 50.,  10.,  10.,  10.,  10.,  10.,  10.,  10.,  10., 100.

In [64]:
def ramp_factory(y1, y2, t1, t2):
    def ramp(t):
        if t<t1:
            return y1
        if t>t2:
            return y2
        return y1+(y2-y1)*(t-t1)/(t2-t1)
    return ramp

In [73]:
ramp = ramp_factory(2, 5, 10, 30)

In [74]:
def rhs(t, vec):

    R = ramp(t)
    M = vec
    L = c*np.where(M<M0, 0, (M-M0))**1.5
    D = L[0]/(R+1)

    Vrec = (R+1)*Dsp
    Vstrip = Vrec - (1-q)*F

    dM = np.zeros_like(M)
    dM -= L
    dM[1:]+=L[:-1]
    dM[1]-=Dsp

    dM[:NFeed]+=Vrec
    dM[NFeed:-1]+=Vstrip

    dM[1:NFeed+1]-=Vrec
    dM[NFeed+1:]-=Vstrip

    dM[NFeed]+=q*F
    dM[NFeed-1]+=(1-q)*F

    return dM


In [66]:
tend =100
tplot = np.linspace(0,tend, 200)
res = sp.integrate.solve_ivp(rhs, (0,tend), y0=Minitial, method='Radau', dense_output=True)
Ms = res.sol(tplot)

In [71]:
Rplot = np.array([ramp(t) for t in tplot])

In [72]:
fig=make_subplots(rows=1,cols=2)
for i, M in enumerate(Ms):
    fig.add_scatter(x=tplot, y=M, name=f'{i}', row=1, col=1)
fig.add_scatter(x=tplot, y=Rplot, row=1, col=2)
fig.update_layout(width=600, height=400)

In [ ]:
def rhs2(t, vec):

    R = ramp(t)
    x, M = np.split(vec,2)
    y = alpha*x/(1-x+alpha*x)

    L = c*np.where(M<M0, 0, (M-M0))**1.5
    D = L[0]/(R+1)

    Vrec = (R+1)*Dsp
    Vstrip = Vrec - (1-q)*F

    dM = np.zeros_like(M)
    dxM = np.zeros_like(x)
    dM -= L
    dxM -= x*L
    dM[1:]+=L[:-1]
    dxM[1:]+=x[:-1]*L[:-1]
    dM[1]-=Dsp
    dxM[1]-=x[0]*Dsp

    dM[:NFeed]+=Vrec
    dxM[:NFeed]+=y[:NFeed]*Vrec
    dM[NFeed:-1]+=Vstrip

    dM[1:NFeed+1]-=Vrec
    dM[NFeed+1:]-=Vstrip

    dM[NFeed]+=q*F
    dM[NFeed-1]+=(1-q)*F

    return dM

In [76]:
x_initial = np.full_like(M0, zF)

In [ ]:
tend =100
tplot = np.linspace(0,tend, 200)
res = sp.integrate.solve_ivp(rhs, (0,tend), y0=np.r_[x_initial, Minitial], method='Radau', dense_output=True)
Ms = res.sol(tplot)